In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 50)

orders   = pd.read_csv("Desktop/DATA ANALYST/DA PROJECT/e-commerce-profitability-analysis-kvrti/brightcart/data_raw/orders.csv", parse_dates=["order_date"])
products = pd.read_csv("Desktop/DATA ANALYST/DA PROJECT/e-commerce-profitability-analysis-kvrti/brightcart/data_raw/products.csv")
mkt      = pd.read_csv("Desktop/DATA ANALYST/DA PROJECT/e-commerce-profitability-analysis-kvrti/brightcart/data_raw/marketing_spend.csv")

for name, df in [("orders", orders), ("products", products), ("mkt", mkt)]:
    print("=====", name, df.shape)
    print(df.dtypes)
    print("nulls:", int(df.isna().sum().sum()), "| duplicate rows:", int(df.duplicated().sum()))

print(orders["channel"].value_counts())
print(orders["primary_category"].value_counts())
print(orders["returned"].value_counts())
print(mkt["platform"].value_counts())
print(orders["order_date"].min(), orders["order_date"].max())


===== orders (2000, 23)
order_id                    object
customer_id                 object
order_date          datetime64[ns]
channel                     object
payment_method              object
region                      object
items_ordered                int64
primary_category            object
gross_revenue              float64
discount_pct                 int64
discount_amount            float64
shipping_cost              float64
product_cost               float64
platform_fee               float64
transaction_fee            float64
returned                    object
refund_amount              float64
net_revenue                float64
total_costs                float64
profit                     float64
order_month                 object
order_year                   int64
is_returned                  int64
dtype: object
nulls: 0 | duplicate rows: 0
===== products (207, 9)
product_id                 object
product_name               object
category                   object
su

In [4]:
# 1) Trim stray spaces in text columns (cheap insurance)
for c in ["channel", "primary_category", "region", "payment_method", "returned"]:
    orders[c] = orders[c].str.strip()
mkt["platform"] = mkt["platform"].str.strip()
products["category"] = products["category"].str.strip()

# 2) Helper columns
orders["order_month"] = orders["order_date"].dt.to_period("M").astype(str)   # e.g. 2025-06
orders["order_year"]  = orders["order_date"].dt.year
orders["is_returned"] = (orders["returned"] == "Yes").astype(int)

# 3) VALIDATE the identities (these should all pass)
err1 = (orders.gross_revenue - orders.discount_amount - orders.refund_amount - orders.net_revenue).abs().max()
err2 = (orders.product_cost + orders.shipping_cost + orders.platform_fee + orders.transaction_fee - orders.total_costs).abs().max()
err3 = (orders.net_revenue - orders.total_costs - orders.profit).abs().max()
print(err1, err2, err3)
assert err1 < 0.02 and err2 < 0.02 and err3 < 0.02

# 4) Refunds only exist on returned orders
print(orders.loc[orders.returned == "No", "refund_amount"].sum())    # expect 0

# 5) Business-rule check: platform fees by channel
print(orders.groupby("channel")["platform_fee"].agg(["min", "max", "mean"]))

# 6) Marketing: confirm ROAS column = attributed revenue / spend
print((mkt.roas - mkt.revenue_attributed / mkt.spend).abs().max())   # tiny rounding only

# 7) Save cleaned copies
orders.to_csv("Desktop/DATA ANALYST/DA PROJECT/e-commerce-profitability-analysis-kvrti/brightcart/data_raw/orders.csv", index=False)
products.to_csv("Desktop/DATA ANALYST/DA PROJECT/e-commerce-profitability-analysis-kvrti/brightcart/data_raw/products.csv", index=False)
mkt.to_csv("Desktop/DATA ANALYST/DA PROJECT/e-commerce-profitability-analysis-kvrti/brightcart/data_raw/marketing_spend.csv", index=False)


1.1368683772161603e-13 5.684341886080802e-14 1.1368683772161603e-13
0.0
                 min    max  mean
channel                          
Marketplace     0.82 162.17 18.97
Mobile App      0.00   0.00  0.00
Social Commerce 0.62  63.13  9.87
Website         0.00   0.00  0.00
0.004991512673118592


In [5]:

# Checkpoint validation

actual = {
    "Orders": len(orders),
    "Gross revenue": orders["gross_revenue"].sum(),
    "Discounts": orders["discount_amount"].sum(),
    "Refunds": orders["refund_amount"].sum(),
    "Net revenue": orders["net_revenue"].sum(),
    "Total profit": orders["profit"].sum(),
    "Marketing spend (all 24 months)": mkt["spend"].sum()
}

expected = {
    "Orders": 2000,
    "Gross revenue": 277969.13,
    "Discounts": 21068.31,
    "Refunds": 20582.45,
    "Net revenue": 236318.37,
    "Total profit": 56333.83,
    "Marketing spend (all 24 months)": 503506.14
}

for metric, expected_value in expected.items():
    actual_value = actual[metric]
    difference = actual_value - expected_value

    print(
        f"{metric}: Actual = {actual_value:,.2f} | "
        f"Expected = {expected_value:,.2f} | "
        f"Difference = {difference:,.2f}"
    )


Orders: Actual = 2,000.00 | Expected = 2,000.00 | Difference = 0.00
Gross revenue: Actual = 277,969.13 | Expected = 277,969.13 | Difference = 0.00
Discounts: Actual = 21,068.31 | Expected = 21,068.31 | Difference = 0.00
Refunds: Actual = 20,582.45 | Expected = 20,582.45 | Difference = 0.00
Net revenue: Actual = 236,318.37 | Expected = 236,318.37 | Difference = 0.00
Total profit: Actual = 56,333.83 | Expected = 56,333.83 | Difference = 0.00
Marketing spend (all 24 months): Actual = 503,506.14 | Expected = 503,506.14 | Difference = 0.00
